# **GPT**

In [ ]:
import json
import re
from collections import Counter


def _normalize_token(x):
    if x is None:
        return ""
    s = str(x)
    s = s.strip()
    s = re.sub(r'\s+', ' ', s)
    return s.lower()


def _norm_quad_as_tuple(q):
    """Normalize and convert a quadruple list into a tuple."""
    if not isinstance(q, (list, tuple)):
        return None
    return tuple(_normalize_token(el) for el in q)


def quads_to_counter(quads):
    """Return a Counter (multiset) of normalized quadruples."""
    c = Counter()
    if not isinstance(quads, list):
        return c
    for q in quads:
        tq = _norm_quad_as_tuple(q)
        if tq is not None:
            c[tq] += 1
    return c


def find_mismatched_predictions(input_filename, output_filename):
    """
    Reads a JSON Lines file, compares the 'pred' and 'ref' tuples for each entry
    as multisets (respecting duplicates, with normalization), and writes all
    entries where the predictions do not match the references to a new file.

    Args:
        input_filename (str): Input JSONL file path.
        output_filename (str): Output JSONL file path.
    """
    mismatched_entries = []
    try:
        with open(input_filename, 'r', encoding='utf-8') as infile:
            for line_number, line in enumerate(infile, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    data = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Skipping line {line_number} due to JSON decode error.")
                    continue

                pred_counter = quads_to_counter(data.get('pred', []))
                ref_counter = quads_to_counter(data.get('ref', []))

                if pred_counter != ref_counter:
                    mismatched_entries.append(data)

        with open(output_filename, 'w', encoding='utf-8') as outfile:
            for entry in mismatched_entries:
                outfile.write(json.dumps(entry, ensure_ascii=False) + '\n')

        print(f"Successfully processed {len(mismatched_entries)} mismatched entries.")
        print(f"Mismatched entries saved to '{output_filename}'.")

    except FileNotFoundError:
        print(f"Error: The file '{input_filename}' was not found.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


if __name__ == "__main__":
    input_file = "/content/test_predictions - Rest15.jsonl"  # adjust path as needed
    output_file = "mismatched_predictions.jsonl"
    find_mismatched_predictions(input_file, output_file)


Successfully processed 311 mismatched entries.
Mismatched entries saved to 'mismatched_predictions.jsonl'.


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "" # Replace with your actual key

In [ ]:
from getpass import getpass
import os
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")


OpenAI API key: ··········


In [ ]:
#!/usr/bin/env python3
"""
Enhanced re-evaluation script using a judge model (e.g., o3) to arbitrate mismatches.

Key enhancements vs. previous version:
- Neutralizes bias: the judge is not told directly which is "prediction" vs. "ground truth".
- Judge outputs compact label: 0 → favor first tuple set, 1 → favor second tuple set.
- Supports reasoning capture.
- Updates TP/FP/FN accordingly if model tuples are favored.
- Saves detailed logs and final metrics.
"""

import os
import json
import argparse
import random
import time
from typing import List, Set, Tuple, Dict

try:
    from openai import OpenAI
except Exception:
    import openai as _openai
    OpenAI = None


def normalize_quad(q: List) -> Tuple:
    if not isinstance(q, (list, tuple)) or len(q) != 4:
        return None
    return tuple(str(x).strip().lower() for x in q)


def quads_to_set(quads: List[List]) -> Set[Tuple]:
    if not quads:
        return set()
    return {nq for q in quads if (nq := normalize_quad(q)) is not None}


def query_judge_model(prompt: str, model_name: str, client, max_retries: int = 2) -> Dict:
    for attempt in range(1, max_retries + 1):
        try:
            if OpenAI is not None and isinstance(client, OpenAI):
                resp = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    # temperature=0.0,
                )
                content = resp.choices[0].message.content
            else:
                _openai.api_key = client
                resp = _openai.ChatCompletion.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    # temperature=0.0,
                )
                content = resp.choices[0].message["content"]

            try:
                return json.loads(content)
            except json.JSONDecodeError:
                start, end = content.find("{"), content.rfind("}")
                if start != -1 and end != -1 and end > start:
                    try:
                        return json.loads(content[start:end+1])
                    except Exception:
                        pass
                return {"decision": 1, "reasoning": "Invalid JSON from judge", "raw": content}
        except Exception as e:
            print(f"[Judge call] Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                time.sleep(attempt)
            else:
                return {"decision": 1, "reasoning": f"Error during API call: {e}"}


def build_prompt(sentence: str, dependency_tree: str, option0, option1) -> str:
    template = """### ROLE: Expert Linguistic Analyst and Judge

### OBJECTIVE:
You must evaluate which of two candidate extractions of aspect-sentiment quadruples is more correct, based strictly on the rules below.

### TASK DEFINITION
A valid quadruple = [Aspect, Category, Sentiment, Opinion Word]

**RULES:**
1. All elements must be grounded in the query sentence.
2. Opinion Word must appear verbatim in the sentence.
3. Aspect & Category must be logically consistent. Category must be one of:
   - ambience general
   - drinks prices
   - drinks quality
   - drinks style_options
   - food prices
   - food quality
   - food style_options
   - location general
   - restaurant general
   - restaurant miscellaneous
   - restaurant prices
   - service general
4. Sentiment must accurately reflect the tone of the Opinion Word.

### CONTEXT:
- Query Sentence: {sentence}
- Dependency Tree: {dependency_tree}

### OPTIONS TO JUDGE:
Option 0: {opt0}
Option 1: {opt1}

### ASSIGNMENT:
Determine which option is more accurate according to the rules.

Return only JSON with keys "reasoning" and "decision".
- reasoning: explanation for your judgment
- decision: 0 if Option 0 is correct, 1 if Option 1 is correct
"""
    return template.format(
        sentence=sentence,
        dependency_tree=dependency_tree,
        opt0=json.dumps(option0, ensure_ascii=False, indent=2),
        opt1=json.dumps(option1, ensure_ascii=False, indent=2),
    )


def re_evaluate(predictions_file: str, mismatched_file: str, model_name: str, limit: int):
    def compute_initial_metrics(path: str):
        tp = fp = fn = 0
        total = 0
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                total += 1
                data = json.loads(line)
                pred_set = quads_to_set(data.get("pred", []))
                ref_set = quads_to_set(data.get("ref", []))
                tp += len(pred_set & ref_set)
                fp += len(pred_set - ref_set)
                fn += len(ref_set - pred_set)
        return tp, fp, fn, total

    tp, fp, fn, total = compute_initial_metrics(predictions_file)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    print(f"Original: samples={total} TP={tp} FP={fp} FN={fn} P={precision:.4f} R={recall:.4f} F1={f1:.4f}")

    if not os.path.exists(mismatched_file):
        with open(predictions_file, "r", encoding="utf-8") as fin, open(mismatched_file, "w", encoding="utf-8") as fout:
            for line in fin:
                data = json.loads(line)
                if quads_to_set(data.get("pred", [])) != quads_to_set(data.get("ref", [])):
                    fout.write(json.dumps(data, ensure_ascii=False) + "\n")

    with open(mismatched_file, "r", encoding="utf-8") as f:
        mismatched_samples = [json.loads(line) for line in f]

    if limit and 0 < limit < len(mismatched_samples):
        random.shuffle(mismatched_samples)
        mismatched_samples = mismatched_samples[:limit]

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise EnvironmentError("OPENAI_API_KEY not set")
    if OpenAI is not None:
        client = OpenAI(api_key=api_key)
    else:
        client = api_key

    new_tp, new_fp, new_fn = tp, fp, fn
    judged_logs = []

    for idx, sample in enumerate(mismatched_samples, 1):
        print(f"[{idx}/{len(mismatched_samples)}]")
        user_content = sample.get("input", {}).get("text", "")
        if "Depency-tree of query:" not in user_content:
            continue
        try:
            query_part, dep_tree_part = user_content.split("Depency-tree of query:", 1)
            sentence = query_part.replace("Query:", "").strip()
            dep_tree = dep_tree_part.replace("LABELS:", "").strip()
        except Exception:
            continue

        prompt = build_prompt(sentence, dep_tree, sample.get("pred", []), sample.get("ref", []))
        verdict = query_judge_model(prompt, model_name, client)

        judged_logs.append({"sample": sample, "verdict": verdict})

        if verdict.get("decision") == 0:  # favor prediction
            pred_set = quads_to_set(sample.get("pred", []))
            ref_set = quads_to_set(sample.get("ref", []))
            new_tp += len(pred_set - ref_set)
            new_fp -= len(pred_set - ref_set)
            new_fn -= len(ref_set - pred_set)

    new_precision = new_tp / (new_tp + new_fp) if (new_tp + new_fp) else 0.0
    new_recall = new_tp / (new_tp + new_fn) if (new_tp + new_fn) else 0.0
    new_f1 = 2 * new_precision * new_recall / (new_precision + new_recall) if (new_precision + new_recall) else 0.0

    summary = {
        "original": {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1},
        "re_evaluated": {"tp": new_tp, "fp": new_fp, "fn": new_fn, "precision": new_precision, "recall": new_recall, "f1": new_f1},
        "judged_samples": len(judged_logs),
    }

    with open("re-evaluation_summary.json", "w", encoding="utf-8") as fout:
        json.dump(summary, fout, ensure_ascii=False, indent=2)
    with open("judged_samples_details.jsonl", "w", encoding="utf-8") as fout:
        for entry in judged_logs:
            fout.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"Re-evaluated: TP={new_tp} FP={new_fp} FN={new_fn} P={new_precision:.4f} R={new_recall:.4f} F1={new_f1:.4f}")

    print("Re-evaluation complete. Summary written to re-evaluation_summary.json")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--predictions", default="/content/test_predictions - Rest15.jsonl")
    parser.add_argument("--mismatched", default="/content/mismatched_predictions.jsonl")
    parser.add_argument("--model", default="o3", help="Judge model name")
    parser.add_argument("--limit", type=int, default=None)
    args, _ = parser.parse_known_args()
    re_evaluate(args.predictions, args.mismatched, args.model, args.limit)


Original: samples=537 TP=420 FP=350 FN=352 P=0.5455 R=0.5440 F1=0.5447
[1/311]
[2/311]
[3/311]
[4/311]
[5/311]
[6/311]
[7/311]
[8/311]
[9/311]
[10/311]
[11/311]
[12/311]
[13/311]
[14/311]
[15/311]
[16/311]
[17/311]
[18/311]
[19/311]
[20/311]
[21/311]
[22/311]
[23/311]
[24/311]
[25/311]
[26/311]
[27/311]
[28/311]
[29/311]
[30/311]
[31/311]
[32/311]
[33/311]
[34/311]
[35/311]
[36/311]
[37/311]
[38/311]
[39/311]
[40/311]
[41/311]
[42/311]
[43/311]
[44/311]
[45/311]
[46/311]
[47/311]
[48/311]
[49/311]
[50/311]
[51/311]
[52/311]
[53/311]
[54/311]
[55/311]
[56/311]
[57/311]
[58/311]
[59/311]
[60/311]
[61/311]
[62/311]
[63/311]
[64/311]
[65/311]
[66/311]
[67/311]
[68/311]
[69/311]
[70/311]
[71/311]
[72/311]
[73/311]
[74/311]
[75/311]
[76/311]
[77/311]
[78/311]
[79/311]
[80/311]
[81/311]
[82/311]
[83/311]
[84/311]
[85/311]
[86/311]
[87/311]
[88/311]
[89/311]
[90/311]
[91/311]
[92/311]
[93/311]
[94/311]
[95/311]
[96/311]
[97/311]
[98/311]
[99/311]
[100/311]
[101/311]
[102/311]
[103/311]
[104/31